# Live host telemetry evaluation

This notebook fetches authenticated observations from the running platform and calls the same forecasting and anomaly modules used by the API. It contains no synthetic measurements. Forecast errors and anomaly counts describe only the selected host and returned history. They are not labeled failure-prediction metrics.

In [ ]:
import getpass
import json
import os
import sys
from pathlib import Path
from urllib.request import Request, urlopen

ROOT = Path.cwd()
if not (ROOT / 'backend' / 'app').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'backend'))
API = os.getenv('LAP_API_URL', 'http://127.0.0.1:8000/api').rstrip('/')
username = os.getenv('LAP_USERNAME') or input('API username: ')
password = os.getenv('LAP_PASSWORD') or getpass.getpass('API password: ')

def api_json(path, token=None, method='GET', body=None):
    headers = {'Content-Type': 'application/json'}
    if token:
        headers['Authorization'] = f'Bearer {token}'
    payload = json.dumps(body).encode() if body is not None else None
    request = Request(f'{API}{path}', data=payload, headers=headers, method=method)
    with urlopen(request, timeout=20) as response:
        return json.load(response)

session = api_json('/auth/login', method='POST', body={'username': username, 'password': password})
token = session['token']
rows = api_json('/system/metrics/history?limit=5000', token=token)
print(f'Retrieved {len(rows)} actual stored observations from {API}')

In [ ]:
from app.forecasting import forecast_resources
from app.intelligence import detect_anomalies

forecasts = forecast_resources(rows)
for item in forecasts:
    print({key: item.get(key) for key in ('metric', 'status', 'sample_count', 'method', 'walk_forward_mae', 'baseline_mae', 'evaluation_count', 'estimated_hours_to_threshold')})

anomaly_results = detect_anomalies(rows)
flagged = [item for item in anomaly_results if item.get('is_anomaly')]
print(f'Anomalies flagged by the current unsupervised window: {len(flagged)} of {len(anomaly_results)} observations')
for item in flagged[-20:]:
    print({key: item.get(key) for key in ('timestamp', 'dominant_metric', 'observed_value', 'baseline_value', 'detection_method', 'anomaly_score', 'explanation')})

## Interpretation

Forecast MAE is measured chronologically against held-out observations and compared with the last-value baseline. The anomaly detector is unsupervised; flagged points are unusual relative to the supplied window and are not confirmed failures. Do not compare anomaly counts across hosts as accuracy. Save results outside the notebook only if the host, collection interval, date range, and software version are recorded. No outputs or credentials are committed with this notebook.